In [16]:
# Utility notebook containing a collection of documented plotting and
# analysis functions used across the project for atmospheric data
# exploration and visualization.
#
# The notebook includes:
# - `doanes_rule_bins` for computing histogram bin edges using Doane’s rule
# - `cdf_even_odd` for comparing CDFs between even/odd indexed subsets
# - `cdf_comparison` for general CDF comparisons between data subsets
# - `doane_hist` for histogram and CDF visualization using Doane binning
# - `plot_median_per_bin_doane` for computing and plotting median values
#   within Doane-binned intervals
# - `month_year_boxplot` for seasonal and yearly box-and-whisker analysis
#
# Each function is documented within the notebook cells and is designed
# for reuse in downstream analysis and visualization workflows.

from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew
import math
from scipy.stats import ks_2samp
from scipy.stats import binned_statistic

### Calculate number of bins for a column of data (Using Doane's Rule)

In [2]:
# Calculate number of bins using Doane's rule

def doanes_rule_bins(data):
    
    n = len(data)
    g1 = skew(data)
    sigma_g1 = np.sqrt((6 * (n - 2)) / ((n + 1) * (n + 3)))
    
    return int(np.round(1 + np.log2(n) + np.log2(1 + np.abs(g1) / sigma_g1)))

### Create CDFs constructed from even and odd rows in a column from a data file

In [5]:
# Plot a CDF that represents the even rows of a column of data 
# Plot a CDF that represents the odd rows of a column of data 
# Perform a KS test to compare the two CDFs
# Title : str (for the histogram (i.e. 'DIMM', 'MASS', 'Ground Layer'))

def cdf_even_odd(data, title):

    # Compute even and odd rows
    even_data = data[::2]
    odd_data = data[1::2]

    # Apply the function to calculate number of bins using Doane's Rule
    bins_even = doanes_rule_bins(even_data)
    bins_odd = doanes_rule_bins(odd_data)

    # Compute number of data points (counts) that fall into each bin and bin edges
    counts_even, bin_edges_even = np.histogram(even_data, bins=bins_even)
    counts_odd, bin_edges_odd = np.histogram(odd_data, bins=bins_odd)

    # Compute center of each bin
    bin_centers_even = (bin_edges_even[:-1] + bin_edges_even[1:]) / 2
    bin_centers_odd = (bin_edges_odd[:-1] + bin_edges_odd[1:]) / 2

    # Compute the cumulative sum of counts
    cdf_even = np.cumsum(counts_even)
    cdf_odd = np.cumsum(counts_odd)

    # Scale the CDF so that it goes from 0 to 1
    # Turn it into a probability-based cumulative distribution function
    cdf_even = cdf_even / cdf_even[-1]
    cdf_odd = cdf_odd / cdf_odd[-1]

    # Perform KS test
    ks_stat, p_value = ks_2samp(even_data, odd_data)

    # Print the KS test values
    # D : Max vertical distance between both CDFs
    # p : 1 = Exactly the same, 0 = Completely different 
    print(f'KS Test: D = {ks_stat:.3f}, p = {p_value:.3f}')

    # Plot the CDFs 
    plt.figure(figsize=(6, 4))

    plt.plot(bin_centers_even, cdf_even, color='darkred', linewidth=4, linestyle='--', label='Even Values')
    plt.plot(bin_centers_odd, cdf_odd, color='lightblue', linewidth=2, label='Odd Values')

    plt.title(f'{title} Seeing CDF (Even vs. Odd Data)', fontsize = 14)
    plt.xlabel(f'{title} Value', fontsize = 12)
    plt.ylabel('Cumulative Probability', fontsize = 12)

    plt.ylim(0,1.05)

    plt.legend()
    plt.tight_layout()
    plt.show()

### Create CDFs constructed from two different columns in one or more data file(s)

In [6]:
# Plot a CDF that represents a column of data from the first data file
# Plot a CDF that represents a column of data from the second data file
# Perform a KS test to compare the two CDFs
# Title1 : str (first set of values to compare (i.e. 'DIMM', 'MASS', 'Ground Layer'))
# Title2 : str (second set of values to compare (i.e. 'DIMM', 'MASS', 'Ground Layer'))

def cdf_comparison(data1, data2, title1, title2):

    # Apply the function to calculate number of bins using Doane's Rule
    bins1 = doanes_rule_bins(data1)
    bins2 = doanes_rule_bins(data2)

    # Compute number of data points (counts) that fall into each bin and bin edges
    counts1, bin_edges1 = np.histogram(data1, bins=bins1)
    counts2, bin_edges2 = np.histogram(data2, bins=bins2)

    # Compute center of each bin
    bin_centers1 = (bin_edges1[:-1] + bin_edges1[1:]) / 2
    bin_centers2 = (bin_edges2[:-1] + bin_edges2[1:]) / 2

    # Compute the cumulative sum
    cdf1 = np.cumsum(counts1)
    cdf2 = np.cumsum(counts2)

    # Scale the CDF so that it goes from 0 to 1
    # Turn it into a probability-based cumulative distribution function
    cdf1 = cdf1 / cdf1[-1]
    cdf2 = cdf2 / cdf2[-1]

    # Perform KS test
    ks_stat, p_value = ks_2samp(data1, data2)

    # Print the KS test values. 
    # D : Max vertical distance between both CDFs
    # p : 1 = Exactly the same, 0 = Completely different
    print(f'KS Test: D = {ks_stat:.3f}, p = {p_value:.3f}')
    
    # Plot the CDFs
    plt.figure(figsize=(6, 4))

    plt.plot(bin_centers1, cdf1, color='darkred', linewidth=2, label=f'{title1}')
    plt.plot(bin_centers2, cdf2, color='darkblue', linewidth=2, label=f'{title2}')

    plt.title(f'{title1} vs {title2} Seeing CDFs', fontsize = 14)
    plt.xlabel('Value', fontsize = 12)
    plt.ylabel('Cumulative Probability', fontsize = 12)

    plt.ylim(0,1.05)

    plt.legend()
    plt.tight_layout()
    plt.show()

### Create CDFs constructed from first and second half of rows in a column from a data file

In [7]:
# Plot a CDF that represents the first half of rows of a column of data 
# Plot a CDF that represents the second half of rows of a column of data 
# Perform a KS test to compare the two CDFs
# Title : str (for the histogram (i.e. 'DIMM', 'MASS', 'Ground Layer'))
# Zoom : bool (Determine whether to plot a zoomed region of the CDF or not)

def cdf_first_second_half(data, title, zoom = True):

    # Split data in half
    half = len(data) // 2
    first_half = data[:half]
    second_half = data[half:]

    # Apply the function to calculate number of bins using Doane's Rule
    bins_first = doanes_rule_bins(first_half)
    bins_second = doanes_rule_bins(second_half)

    # Compute number of data points (counts) that fall into each bin and bin edges
    counts_first, bin_edges_first = np.histogram(first_half, bins=bins_first)
    counts_second, bin_edges_second = np.histogram(second_half, bins=bins_second)

    # Compute center of each bin
    bin_centers_first = (bin_edges_first[:-1] + bin_edges_first[1:]) / 2
    bin_centers_second = (bin_edges_second[:-1] + bin_edges_second[1:]) / 2

    # Compute the cumulative sum
    cdf_first = np.cumsum(counts_first)
    cdf_second = np.cumsum(counts_second)

    # Scale the CDF so that it goes from 0 to 1
    # Turn it into a probability-based cumulative distribution function
    cdf_first = cdf_first / cdf_first[-1]
    cdf_second = cdf_second / cdf_second[-1]

    # Perform KS test
    ks_stat, p_value = ks_2samp(first_half, second_half)

    # Print the KS test values. 
    # D : Max vertical distance between both CDFs
    # p : 1 = Exactly the same, 0 = Completely different
    print(f'KS Test: D = {ks_stat:.3f}, p = {p_value:.3f}')

    # Zoom in on specifc region of plot (Between 0 and 1)
    if zoom: 
        # Plot the CDFs 
        plt.figure(figsize=(6, 4))
        
        plt.plot(bin_centers_first, cdf_first, color='darkred', linewidth=4, linestyle='--', label='1st Half of Data')
        plt.plot(bin_centers_second, cdf_second, color='lightblue', linewidth=2, label='2nd Half of Data')
        
        plt.title(f'{title} Seeing CDF (First vs. Second Half of Data)', fontsize = 14)
        plt.xlabel(f'{title} Value', fontsize = 12)
        plt.ylabel('Cumulative Probability', fontsize = 12)

        plt.xlim(0,1)
        plt.ylim(0,1.05)
        
        plt.legend()
        plt.tight_layout()
        plt.show()
        
    # Otherwise Plot entire CDF
    else:
        plt.figure(figsize=(6, 4))
        
        plt.plot(bin_centers_first, cdf_first, color='darkred', linewidth=4, linestyle='--', label='1st Half of Data')
        plt.plot(bin_centers_second, cdf_second, color='lightblue', linewidth=2, label='2nd Half of Data')
        
        plt.title(f'{title} Seeing CDF (First vs. Second Half of Data)', fontsize = 14)
        plt.xlabel(f'{title} Value', fontsize = 12)
        plt.ylabel('Cumulative Probability', fontsize = 12)

        plt.ylim(0,1.05)
        
        plt.legend()
        plt.tight_layout()
        plt.show()

### Create a histogram and CDF constructed from a column in a data file

In [22]:
# Plot a histogram that represents a column of data
# Plot a CDF that represents a column of data
# Plot the median from the column of data 
# Title : str (for the histogram (i.e. 'DIMM', 'MASS', 'Ground Layer'))

def doane_hist(data, title, colname, log_plot = True):

    # Apply the function to calculate number of bins using Doane's Rule
    num_bins = doanes_rule_bins(data)

    # Compute number of data points (counts) that fall into each bin and bin centers
    counts, bin_edges = np.histogram(data, bins=num_bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Compute the cumulative sum
    cdf = np.cumsum(counts)

    # Scale the CDF so that it goes from 0 to 1
    # Turn it into a probability-based cumulative distribution function
    cdf = cdf / cdf[-1]

    # Print the max, min, median and standard deviation of data
    print("The max value is: ",round(max(data),4))
    print("The min value is: ", round(min(data),4))
    print("The median value is: ", round(np.median(data),4))
    print("The std value is: ", round(np.std(data),4))
    print("-------------")

    # Print max/min of bins and number of values per bin
    # for i in range(len(counts)):
    #     print(f"Bin {i+1}: Range ({bin_edges[i]:.2f}, {bin_edges[i+1]:.2f}) -> {counts[i]} entries")

    median = np.percentile(data, 50)

    # Plot the log(frequency) histogram
    if log_plot: 
        plt.figure(figsize = (8,5))
        plt.hist(data, bins=bin_edges, color='lightblue', edgecolor='black', alpha=0.6, log=True)
        plt.xlabel(f'{title}', fontsize = 12)
        plt.ylabel("Log (Frequency)")
        plt.title("Histogram with Logarithmic Y-axis (Doane's Rule)")
        plt.show()

    fig, ax1 = plt.subplots(figsize=(8, 5))

    # Histogram on primary axis
    ax1.hist(data, bins=bin_edges, color='lightblue', edgecolor='black', alpha=0.6)
    ax1.set_xlabel(f'{title}', fontsize = 12)
    ax1.set_ylabel('Frequency', color='blue', fontsize = 12)
    ax1.tick_params(axis='y', labelcolor='blue')

    # CDF on secondary axis
    ax2 = ax1.twinx()
    ax2.plot(bin_edges[1:], cdf, color='darkred', linewidth=2)
    ax2.set_ylabel('Cumulative Probability', color='darkred', fontsize = 12)
    ax2.tick_params(axis='y', labelcolor='darkred')
    ax2.set_ylim(0, 1.05)  # important to cap y-axis of CDF at 1

    # Draw percentile lines
    ax2.axvline(median, color='green', linestyle='--', linewidth=2)
    ax2.axhline(0.5, color='purple', linestyle='--', linewidth=2)

    if (colname == 'ground layer'):
        plt.xlim(0,2)
    # Title
    plt.title("Histogram with CDF Overlay (Doane's Rule)", fontsize = 14)
    fig.tight_layout()
    plt.show()

### Plot median of one column of data per bin of a different column of data 

In [3]:
# Plots the median of y values per bin of x using Doane's rule to determine bin count.
# Determine number of bins using Doane's rule

def plot_median_per_bin_doane(x, y, xlabel, ylabel):
    
    num_bins = doanes_rule_bins(x)

    # Compute binned statistics
    median, bin_edges, _ = binned_statistic(x, y, statistic='median', bins=num_bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Plotting
    plt.figure(figsize=(8, 5))
    plt.plot(bin_centers, median, marker='o', linestyle='-', color='darkorange')
    plt.xlabel(f'{xlabel}')
    plt.ylabel(f'{ylabel}')
    plt.title(f"{ylabel}\n(per {xlabel} bin / {num_bins} bins total)")
    plt.grid(True)
    plt.show()

### Plot box and whisker plot of a data column

In [15]:
# Plot two box and whisker plots that represents of a specified column of data 
# First plot represents the data per month
# Second plot represents the data per year

def month_year_boxplot(column_name, label):
    
    # Read data file
    table = Table.read("master_file.fits")
    
    # Parse DATE column into datetime objects
    datetimes = [datetime.fromisoformat(str(d)) for d in table["datetime"]]
    
    # Extract month values
    months = [dt.month for dt in datetimes]
    
    # Extract year values
    years = [dt.year for dt in datetimes]
    
    # Mask rows per month
    january_mask = [m == 1 for m in months]
    february_mask = [m == 2 for m in months]
    march_mask = [m == 3 for m in months]
    april_mask = [m == 4 for m in months]
    may_mask = [m == 5 for m in months]
    june_mask = [m == 6 for m in months]
    july_mask = [m == 7 for m in months]
    aug_mask = [m == 8 for m in months]
    sept_mask = [m == 9 for m in months]
    oct_mask = [m == 10 for m in months]
    nov_mask = [m == 11 for m in months]
    dec_mask = [m == 12 for m in months]
    
    # Mask rows per year
    y2010_mask = [y == 2010 for y in years]
    y2011_mask = [y == 2011 for y in years]
    y2012_mask = [y == 2012 for y in years]
    y2013_mask = [y == 2013 for y in years]
    y2014_mask = [y == 2014 for y in years]
    y2015_mask = [y == 2015 for y in years]
    y2016_mask = [y == 2016 for y in years]
    y2017_mask = [y == 2017 for y in years]
    y2018_mask = [y == 2018 for y in years]
    y2019_mask = [y == 2019 for y in years]
    y2020_mask = [y == 2020 for y in years]
    y2021_mask = [y == 2021 for y in years]
    y2022_mask = [y == 2022 for y in years]
    y2023_mask = [y == 2023 for y in years]
    y2024_mask = [y == 2024 for y in years]
    y2025_mask = [y == 2025 for y in years]
    
    # Month Lists
    jan_values = []
    feb_values = []
    mar_values = []
    apr_values = [] 
    may_values = [] 
    jun_values = []
    jul_values = []
    aug_values = [] 
    sep_values = [] 
    oct_values = [] 
    nov_values = [] 
    dec_values = []

    # Year Lists
    y2010_values = []
    y2011_values = []
    y2012_values = []
    y2013_values = [] 
    y2014_values = [] 
    y2015_values = []
    y2016_values = []
    y2017_values = [] 
    y2018_values = [] 
    y2019_values = [] 
    y2020_values = [] 
    y2021_values = []
    y2022_values = []
    y2023_values = []
    y2024_values = []
    y2025_values = []
    
    # Load month lists
    for jan_value in table[column_name][january_mask]:
        jan_values.append(jan_value)
    for feb_value in table[column_name][february_mask]:
        feb_values.append(feb_value)
    for mar_value in table[column_name][march_mask]:
        mar_values.append(mar_value)
    for apr_value in table[column_name][april_mask]:
        apr_values.append(apr_value)
    for may_value in table[column_name][may_mask]:
        may_values.append(may_value)
    for jun_value in table[column_name][june_mask]:
        jun_values.append(jun_value)
    for jul_value in table[column_name][july_mask]:
        jul_values.append(jul_value)
    for aug_value in table[column_name][aug_mask]:
        aug_values.append(aug_value)
    for sep_value in table[column_name][sept_mask]:
        sep_values.append(sep_value)
    for oct_value in table[column_name][oct_mask]:
        oct_values.append(oct_value)
    for nov_value in table[column_name][nov_mask]:
        nov_values.append(nov_value)
    for dec_value in table[column_name][dec_mask]:
        dec_values.append(dec_value)
    
    # Load year lists 
    for y2010_value in table[column_name][y2010_mask]:
        y2010_values.append(y2010_value)
    for y2011_value in table[column_name][y2011_mask]:
        y2011_values.append(y2011_value)
    for y2012_value in table[column_name][y2012_mask]:
        y2012_values.append(y2012_value)
    for y2013_value in table[column_name][y2013_mask]:
        y2013_values.append(y2013_value)
    for y2014_value in table[column_name][y2014_mask]:
        y2014_values.append(y2014_value)
    for y2015_value in table[column_name][y2015_mask]:
        y2015_values.append(y2015_value)
    for y2016_value in table[column_name][y2016_mask]:
        y2016_values.append(y2016_value)
    for y2017_value in table[column_name][y2017_mask]:
        y2017_values.append(y2017_value)
    for y2018_value in table[column_name][y2018_mask]:
        y2018_values.append(y2018_value)
    for y2019_value in table[column_name][y2019_mask]:
        y2019_values.append(y2019_value)
    for y2020_value in table[column_name][y2020_mask]:
        y2020_values.append(y2020_value)
    for y2021_value in table[column_name][y2021_mask]:
        y2021_values.append(y2021_value)
    for y2022_value in table[column_name][y2022_mask]:
        y2022_values.append(y2022_value)
    for y2023_value in table[column_name][y2023_mask]:
        y2023_values.append(y2023_value)
    for y2024_value in table[column_name][y2024_mask]:
        y2024_values.append(y2024_value)
    for y2025_value in table[column_name][y2025_mask]:
        y2025_values.append(y2025_value)
    
    # Prepare data for boxplot
    month_data_to_plot = [jan_values, feb_values, mar_values, apr_values, may_values, jun_values,
                          jul_values, aug_values, sep_values, oct_values, nov_values, dec_values]
    
    year_data_to_plot = [y2010_values, y2011_values, y2012_values, y2013_values, y2014_values, y2015_values, y2016_values, y2017_values, 
                         y2018_values, y2019_values, y2020_values, y2021_values, y2022_values, y2023_values, y2024_values, y2025_values]

    # Make plots
    # By Month
    plt.figure(figsize=(8, 6))
    plt.boxplot(month_data_to_plot, labels=['Jan', 'Feb', 'March', 'April', 'May', 'June',
                                            'July', 'August', 'Sept', 'Oct', 'Nov', 'Dec'], showfliers=False)
    
    plt.title(label + ' Seeing (By Month)', fontsize = 16)
    plt.xlabel('Month', fontsize=14)
    plt.ylabel(label + ' Seeing', fontsize = 14)
    plt.ylim(0,1.6)
    plt.tight_layout()
    plt.savefig('gl_month.png')
    plt.show()

    # By Year
    plt.figure(figsize=(8, 6))
    plt.boxplot(year_data_to_plot, labels=['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
                                           '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025'], showfliers=False)
    
    plt.title(label + ' Seeing (By Year)', fontsize = 16)
    plt.xlabel('Year', fontsize=14)
    plt.ylabel(label + ' Seeing', fontsize = 14)
    plt.ylim(0,1.6)
    plt.tight_layout()
    plt.savefig('gl_year.png')
    plt.show()

    return